<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Machine Learning
### Sommersemester 2026
Prof. Dr. Stefan Goetze

#  Principal Component Analysis (PCA)

In diesem Notebook betrachten wir ein Verfahren, dass dem unüberwachten Lernen (*unsupervised learning*) zuzuordnen ist, nämlich der [Hauptkomponentenzerlegung](https://de.wikipedia.org/wiki/Hauptkomponentenanalyse) (engl. *Principal Component Analysis*, PCA).

Die PCA ein Algorithmus zur Reduzierung der Dimensionalität.

Das bedeutet, dass PCA Linearkombinationen der Merkmale berechnet, die den Datensatz mit möglichtst geringem Informationsverlust beschreiben.
Wir haben die Hauptkomponentenzerlegung bereits im [Notebook zur Datenvorverarbeitung](https://github.com/fhswf/aki-ml/blob/main/u3/06_Datenvorverarbeitung.ipynb) kennen gelernt und auch verwendet.

Wie üblich importieren wir zunächst einige verwendete Bibliotheken.

In [ ]:
%matplotlib inline
%pip install -U kaleido                  
import numpy as np                       # math
import matplotlib.pyplot as plt          # visualisation
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting
from sklearn.decomposition import PCA    # Principal Component Analysis from scikit-learn
from sklearn.datasets import load_digits # for UCI ML hand-written digits dataset
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import cross_val_score # for evaluating the performance of our model
from sklearn.preprocessing import StandardScaler    # for standardizing the data
import plotly.graph_objects as go        # for interactive visualisation
import plotly.io as pio                  # for saving interactive visualisation as a static image
from matplotlib.patches import Ellipse   # for plotting the confidence ellipse
import matplotlib.animation as animation # for creating animations
import glob                              # for file handling
import soundfile as sf                   # for reading audio files
from librosa.feature import mfcc         # for extracting Mel-frequency cepstral coefficients (MFCCs) from audio data
import matplotlib.ticker as ticker       # for customizing the ticks on the axes
import librosa                           # for audio processing
from IPython.display import HTML         # for displaying HTML content in Jupyter notebook

# Erstellung eines einfache Datzensatzes

Um einen Eindruck von der Funktionsweise der PCA zu bekommen, wollen wir uns einen zufällig generierten Datensatz anschauen:

In [ ]:
np.random.seed(42)   # fix a seed (set random generator for reproducibility)
nfeatures = 2
nsize = 200

# generate random data (normal distribution, randn()), randomly transformed (by rand()))
X = np.dot(
        np.random.rand(nfeatures, 2),  # random linear transform
        np.random.randn(2, nsize)      # actual data (normal distribution)
          ).T

# visualise data
plt.scatter(X[:, 0], X[:, 1])
plt.title("Randomly generated 2D data")
plt.xlabel(r"$x$")
plt.ylabel(r"$y$")
plt.axis('equal')

#plt.savefig("pca_2d_data.png", dpi=300)
#plt.savefig("pca_2d_data.svg")

plt.show()

Man sieht in dem Streudiagramm, dass es im vorliegenden Datensatz eine Wechselbeziehung zwischen dem Merkmal $x$ und dem Merkmal $y$ gibt.
In vorherigen Aufgaben haben eine solche Wechselbeziehung bereits ausgenutzt, z.B. um mittels der Linearen Regression die Variable $y$ durch die Variable $x$ zu beschreiben.
Wir haben gesehen, dass man $y$ durch ein lineares Modell annähern kann.

## Hauptkomponentenanalyse (principal component analysis (PCA)) für den gerade erstellten Datensatz durch sklearn

Bei der **Hauptkomponentenzerlegung** wird die Wechselbeziehung zwischen Variablen ebenfalls ausgenutzt, aber nicht mit dem Ziel, ein Merkmal zu beschreiben.
Vielmehr werden die Wechselbeziehungen in Form von Linearkombinantionen der Merkmale (den sog. *Hauptkomponenten* (*engl: principal components*)) *gelernt*, um den kompletten Datensatz mit weniger Informationen beschreiben zu können.

Mit *Scikit-Learn* können wir die Hauptkomponenten wie folgt berechnen:

In [ ]:
# perform principal component analysis (PCA)
pca = PCA(n_components=2)
pca.fit(X)

Die `fit`-Funktion lernt die Hauptkomponenten des Datensatzes $\mathbf{X}$.

Im trainierten PCA-Modell sind die *Eigenvektoren* der [Kovarianzmatrix](https://de.wikipedia.org/wiki/Kovarianzmatrix) unter dem Attribut `components_` zugreifbar, die *Eigenwerte* (engl. *Eigenvalues*) werden als `explained_variance_` abgelegt.

In [ ]:
print(pca.components_)
print(pca.explained_variance_)

### Optional: Etwas mathematischer Hintergrund

Die PCA findet zwei orthogonale Vektoren (Hauptkomponenten) $\mathbf{v}_1, \mathbf{v}_2$, die die Daten maximal in Varianz projezieren.

Diese Vektoren sind Eigenvektoren der [Kovarianzmatrix](https://de.wikipedia.org/wiki/Kovarianzmatrix) $\mathbf{C}$ der Daten $\mathbf{X}$:

$$
\mathbf{C} = \frac{1}{n-1}(\mathbf{X} - \bar{\mathbf{X}})^T (\mathbf{X} - \bar{\mathbf{X}})
$$

$\bar{\mathbf{X}}$ ist dabei die Matrix, in der jede Zeile der **Mittelwert** der jeweiligen Spalte von $\mathbf{X}$ ist. Also: von jedem Merkmal wird der Durchschnittswert abgezogen. Dadurch wird der Datensatz **zentriert**, sodass er den Mittelwert $0$ hat.

$\mathbf{X} - \bar{\mathbf{X}}$: Das ist der **zentrierte Datensatz** – alle Werte sind nun Abweichungen vom Mittelwert.

$(\mathbf{X} - \bar{\mathbf{X}})^T (\mathbf{X} - \bar{\mathbf{X}})$: Hier wird der zentrierte Datensatz mit seiner **Transponierten** multipliziert. Das ergibt eine Matrix, in der **steht, wie stark die Merkmale miteinander zusammenhängen** – das ist genau das, was man **Kovarianz** nennt.

$\frac{1}{n - 1}$: Das ist der **Normierungsfaktor**. Er sorgt dafür, dass die Kovarianzen **im Durchschnitt pro Datenpunkt** berechnet werden (statt alles zu summieren).


Die Eigenwerte $\lambda_1, \lambda_2$ entsprechen der Varianz entlang dieser Richtungen:

$$
\mathbf{C} \mathbf{v}_i = \lambda_i \mathbf{v}_i
$$

Die Hauptkomponenten sind normierte Einheitsvektoren:

$$
\hat{\mathbf{v}}_i = \frac{\mathbf{v}_i}{\|\mathbf{v}_i\|}
$$

Die Eigenvektoren $\mathbf{v}_i$ werden mit $\sqrt{\lambda_i}$ skaliert, um die Streuung der Daten entlang dieser Richtungen zu visualisieren:

$$
\mathbf{u}_i = \sqrt{\lambda_i} \hat{\mathbf{v}}_i
$$

Diese Vektoren geben die Richtung und "Länge" der Hauptachsen der Datenwolke an und sind hier als Pfeile eingezeichnet.

Man kann diese Parameter `pca.components_` und `pca.explained_variance_` die die `fit()`-Funktion berechnet auch leicht selbst berechnen, indem man die Funktionen zur Bestimmung der Kovarianzmatrix ([`np.cov`](https://numpy.org/devdocs/reference/generated/numpy.cov.html)) und der [Singulärwertzerlegung](https://de.wikipedia.org/wiki/Singul%C3%A4rwertzerlegung) ([`np.linalg.svd`](https://numpy.org/doc/2.2/reference/generated/numpy.linalg.svd.html)) von NumPy verwendet:


In [ ]:
kovmat = np.cov(X.T)
u,s,v = np.linalg.svd(kovmat)
print("Eigenvektoren :", v)
print("Eigenwerte:", s)

Die Hauptkomponenten verlaufen in Richtung der Eigenvektoren, die Eigenwerte geben das Maß der Streuung der Datenpunkte entlang dieser Hauptkomponente an.
Im folgenden Diagramm zeigen die Vektorpfeile (generiert durch die Funktion [`quiver()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.quiver.html)) den Verlauf der Hauptkomponenten an. Die Länge der Pfeile spiegelt die Varianz der Hauptkomponente wieder.


In [ ]:
plt.scatter(X[:, 0], X[:, 1], alpha=0.2)

ursprung = (0,0)
pc = []
colors = ['r', 'b', 'y', 'g']
for i in range(len(v)):
    pc.append((v[i][0]*np.sqrt(s[i]), v[i][1]*np.sqrt(s[i])))
    plt.quiver(*ursprung, *pc[i], color=colors[i], scale=3)
    
#plt.savefig("pca_2d_vec.png", dpi=300)
#plt.savefig("pca_2d_vec.svg")

plt.show()

### Optional: Animation zur Analyse der Richtung, die die höchste Varianz in den Daten erklärt

Der folgende Code erzeugt eine rotierende rote Linie, die einen Vektor repräsentiert, welcher eine mögliche Hauptkomponente des Datensatzes (als Punktewolke dargestellt) visualisieren soll.

Für jeden Datenpunkt wird zusättzlich die senkrechte Projektion auf diesen rotierenden Vektor (rote Linie) gezeigt – dargestellt als gestrichelte graue Linien, die die Abstände der Punkte zum Vektor verdeutlichen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML


# Zentriere die Daten
X_centered = X - X.mean(axis=0)

# Figure setup
fig, ax = plt.subplots(figsize=(6,6))

# Achsenlimits mit Puffer
x_min, x_max = np.min(X[:,0]), np.max(X[:,0])
y_min, y_max = np.min(X[:,1]), np.max(X[:,1])
x_pad = (x_max - x_min) * 0.1
y_pad = (y_max - y_min) * 0.1
ax.set_xlim(x_min - x_pad, x_max + x_pad)
ax.set_ylim(y_min - y_pad, y_max + y_pad)
ax.set_aspect('equal')

# Punktwolke
scatter = ax.scatter(X[:,0], X[:,1], alpha=0.5)

# Vektorlänge
max_len = max(x_max - x_min + 2*x_pad, y_max - y_min + 2*y_pad) / 2

# Rotierender Vektor (rot)
line, = ax.plot([], [], 'r-', lw=2)

# Projektionen (graue Linien)
projections = [ax.plot([], [], 'gray', lw=0.8, linestyle='--')[0] for _ in range(nsize)]

# Varianzlinien entlang Vektor (gestrichelt)
var_pos_line, = ax.plot([], [], 'r--', lw=2)
var_neg_line, = ax.plot([], [], 'r--', lw=2)

# Textfeld für aktuelle Varianz
var_text = ax.text(0.95, 0.95, '', transform=ax.transAxes,
                   ha='right', va='top', fontsize=12,
                   bbox=dict(facecolor='white', alpha=0.7))

# Precompute: Winkel & Einheitsvektoren
frames = np.arange(0, 360, 2)
angles_rad = np.deg2rad(frames)
unit_vectors = np.column_stack((np.cos(angles_rad), np.sin(angles_rad)))  # (nframes, 2)

# Projektionen aller Punkte auf alle Richtungen
all_projections = unit_vectors @ X_centered.T  # (nframes, nsize)

# (Optional) Hauptvektor aus PCA für Vergleich
U, S, Vt = np.linalg.svd(X_centered)
pca_vec = Vt[0]
pca_line, = ax.plot([], [], 'b--', lw=2)
pca_label = ax.text(0.05, 0.90, 'Optimalste Richtung', transform=ax.transAxes,
                    ha='left', va='top', fontsize=10, color='blue',
                    bbox=dict(facecolor='white', alpha=0.7))


ax.legend()

def init():
    line.set_data([], [])
    var_pos_line.set_data([], [])
    var_neg_line.set_data([], [])
    pca_line.set_data([], [])
    for proj_line in projections:
        proj_line.set_data([], [])
    var_text.set_text('')
    pca_label.set_text('Optimalste Richtung')
    return [line, var_pos_line, var_neg_line, var_text, pca_line, pca_label] + projections

def update(i):
    angle = angles_rad[i]
    v = unit_vectors[i]

    # Hauptlinie (rotierender Vektor)
    x_line = np.array([-max_len*v[0], max_len*v[0]])
    y_line = np.array([-max_len*v[1], max_len*v[1]])
    line.set_data(x_line, y_line)

    # Projektionen (auf aktuellen Vektor)
    proj_len = all_projections[i]  # Länge der Projektion auf v
    proj_points = np.outer(proj_len, v)

    for idx in range(nsize):
        x_vals = [X[idx,0], proj_points[idx,0] + X.mean(axis=0)[0]]
        y_vals = [X[idx,1], proj_points[idx,1] + X.mean(axis=0)[1]]
        projections[idx].set_data(x_vals, y_vals)

    # Varianz-Linien
    std = np.std(proj_len)
    mean = 0  # Mittelwert der proj_len ist 0 (zentriert)
    
    var_pos = (mean + std) * v
    var_neg = (mean - std) * v
    var_pos_line.set_data([var_pos[0] - max_len*0.05*v[0], var_pos[0] + max_len*0.05*v[0]],
                          [var_pos[1] - max_len*0.05*v[1], var_pos[1] + max_len*0.05*v[1]])
    var_neg_line.set_data([var_neg[0] - max_len*0.05*v[0], var_neg[0] + max_len*0.05*v[0]],
                          [var_neg[1] - max_len*0.05*v[1], var_neg[1] + max_len*0.05*v[1]])

    # Aktuelle Varianz entlang v (korrekt!)
    var_current = np.var(proj_len)
    var_text.set_text(f'Varianz entlang Richtung:\n{var_current:.4f}')

    # zeige feste optimalste PCA-Richtung
    pca_scale = max_len
    pca_line.set_data([-pca_scale*pca_vec[0], pca_scale*pca_vec[0]],
                      [-pca_scale*pca_vec[1], pca_scale*pca_vec[1]])
    pca_label.set_text('Richtung höchster Varianz')

    return [line, var_pos_line, var_neg_line, var_text, pca_line, pca_label] + projections

ani = animation.FuncAnimation(fig, update, frames=len(frames),
                              init_func=init, blit=True, interval=50)
# ani.save("rotation_pca.gif", writer='pillow', fps=30)

HTML(ani.to_jshtml())


### PCA zur Reduktion der Dimensionalität

Bisher haben das PCA-Verfahren genutzt, um die Hauptkomponenten des Datensatzes zu bestimmen.
Im obigen Beispiel hatten wir zwei Variablen und ebenfalls zwei Hauptkomonenten.
Unser ursprünglicher Datensatz hat sich dadurch nicht verändert.

Um Anzahl der Merkmale reduzieren, können wir aber die Ergebnisse der PCA verwenden undem wir die Werte entlang einer oder mehrerer der kleinsten Hauptkomponenten Null setzen.
Geometrisch gesehen, entspricht dies einer Projektion der Datenpunkte auf die dominierenden Hauptkomponenten.
Mit jedem "weglassen" einer Hauptkomponente verliert man zwar einen Teil der Information aus dem Datensatz (mathematisch gesehen entspricht der Informationsgehalt der *Varianz*). Da es sich aber um die Hauptkomponente mit der jeweils geringsten Varianz handelt, ins der Informationsverlust minimal.

Betrachten wir nun, was mit unserem obigen Beispiel passiert, wenn wir den Datensatz von zwei auf eine Variable reduzieren:

In [ ]:
pca = PCA(n_components=1)
pca.fit(X)
X_pca = pca.transform(X)
print("Ursprüngliche Größe:   ", X.shape)
print("UTransformierte Größe:", X_pca.shape)

Der transformierte Datensatz ist nur noch halb so groß.
Um zu sehen, wie die Daten projiziert wurden, kann man die inverse Transformation durchführen und die Daten in so wieder auf den ursprünglichen 2-dimensionalen Raum abbilden:

In [ ]:
X_new = pca.inverse_transform(X_pca)
plt.scatter(X[:, 0], X[:, 1], alpha=0.2)
plt.scatter(X_new[:, 0], X_new[:, 1], c='b', alpha=0.8)
plt.axis('equal')

#plt.savefig("pca_2d_pro.png", dpi=300)
#plt.savefig("pca_2d_pro.svg")

plt.show()
print("Der Datensatz enthält %.1f%c der ursprünglichen Information (Varianz)." % (100*pca.explained_variance_ratio_[0], '%'))

Die hellen (bzw. transparenteren) Punkte sind die Originaldaten, die dunklen Punkte die transformierten Daten.
Man sieht, dass im transformierten Datensatz alle Punkte entlang einer Geraden laufen, auf die die ursprünglichen Punkte projiziert wurden.

Dieser Datensatz mit reduzierter Dimension ist in gewisser Hinsicht *gut genug*, um die wichtigsten Beziehungen zwischen den Punkten zu erfassen: Trotz der Reduzierung der Dimension und damit der Datenmenge um $50\%$ bleibt die Varianz zu fast $95\%$  erhalten.



## Principal Component Analysis (PCA) auf 3D-Daten

Nachdem wir die Hauptkomponentenanalyse an einem zweidimensionalen Datensatz durchgeführt haben, erweitern wir nun das Verfahren auf den dreidimensionalen Raum.

Das Vorgehen bleibt grundsätzlich gleich: Wir erzeugen Zufallsdaten, auf die eine Hauptkomponentenanalyse angewendet wird. Dabei berechnen wir diesmal zwei Hauptkomponenten, die zusammen eine Ebene im 3D-Raum aufspannen.

Im Folgenden erzeugen wir einen Datensatz mit drei Merkmalen (Features).

In [ ]:
# 3D-Daten
nfeatures = 3
nsize = 200

# Erzeuge normalverteilte Zufallsdaten und transformiere sie linear

X_3d = np.dot(
    np.random.rand(nfeatures, 3),   # zufällige lineare Transformation
    np.random.randn(3, nsize)       # Normalverteilte Daten
).T  # shape: (200, 3)

In [ ]:
# 3D-Visualisierung der Daten
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], alpha=0.4)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_zlabel(r"$z$")
plt.title("Originale 3D-Daten")

#plt.savefig("pca_3d_data.png", dpi=300)
#plt.savefig("pca_3d_data.svg")

plt.show()

### Kumulative erklärte Varianz der Hauptkomponenten

Im untigen Plot sehen wir die kumulative erklärte Varianz, die durch die Hauptkomponenten der PCA (Principal Component Analysis) an unseren 3D-Daten erklärt wird. 

Da unsere Daten ursprünglich 3 Dimensionen haben, sind auch maximal 3 Hauptkomponenten möglich. 

Die kumulative erklärte Varianz gibt an, wie viel Information (Varianz) durch die ersten $n$ Hauptkomponenten zusammen abgedeckt wird. Typischerweise möchte man so wenige Hauptkomponenten wie möglich wählen, die aber möglichst viel Varianz erklären.

Dieser Plot hilft dabei, eine sinnvolle Anzahl von Komponenten für eine Dimensionsreduktion auszuwählen, z.B. wenn man die Daten für weitere Analysen oder Visualisierungen vereinfachen möchte.


In [ ]:
X_3d_full = np.copy(X_3d)  

# Zentrieren und normalisieren (z-Score-Normalisierung)
X_3d_full = X_3d_full - np.mean(X_3d_full, axis=0)
X_3d_full = X_3d_full / np.std(X_3d_full, axis=0, ddof=1)

# PCA durchführen
pca = PCA()
pca.fit(X_3d_full)

# Kumulative erklärte Varianz berechnen
exp_var_ratio = pca.explained_variance_ratio_
exp_var_cumul = np.cumsum(exp_var_ratio)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(exp_var_cumul) + 1), exp_var_cumul, marker="o", linestyle="-")
plt.xticks(range(1, len(exp_var_cumul) + 1))
plt.xlabel("Anzahl der Hauptkomponenten")
plt.ylabel("Kumulative erklärte Varianz")
plt.title("PCA: Kumulative erklärte Varianz (alle Features)")
plt.grid(True)
plt.tight_layout()

#plt.savefig("pca_cum_var_3d.png", dpi=300)
#plt.savefig("pca_cum_var_3d.svg")

plt.show()


### Durchführung der PCA im 3D-Raum

Wie bereits im 2D-Fall wenden wir nun die PCA mit `scikit-learn` an. Diesmal berechnen wir drei Hauptkomponenten, da unser Datensatz drei Merkmale enthält.

Durch die Analyse der Varianzanteile, die von den einzelnen Hauptkomponenten erklärt werden, gewinnen wir Einsicht darüber, wie stark die Daten entlang jeder Hauptachse streuen. Dies gibt uns auch Hinweise darauf, ob eine Reduktion auf zwei Komponenten sinnvoll ist.


In [ ]:
pca_3d = PCA(n_components=3)
pca_3d.fit(X_3d)
print("Der Datensatz enthält %.1f%c der Varianz der ersten Komponente." % (100 * pca_3d.explained_variance_ratio_[0], '%'))
print("Der Datensatz enthält %.1f%c der Varianz der zweiten Komponente." % (100 * pca_3d.explained_variance_ratio_[1], '%'))
print("Der Datensatz enthält %.1f%c der Varianz der dritten Komponente." % (100 * pca_3d.explained_variance_ratio_[2], '%'))


### Hauptkomponenten und erklärte Varianz

Nachdem das PCA-Modell auf die 3D-Daten angepasst wurde, können wir die Ergebnisse der Analyse genauer untersuchen.

Die folgenden Ausgaben zeigen:

- die Richtungsvektoren der Hauptkomponenten (also die Achsen, entlang derer die Varianz maximal ist),
- sowie die zugehörigen Varianzen, die entlang dieser Achsen erklärt werden.



In [ ]:
print(pca_3d.components_)
print(pca_3d.explained_variance_)

### Vergleich mit der Eigenwertzerlegung der Kovarianzmatrix

Die Hauptkomponentenanalyse basiert auf der Eigenwertzerlegung der Kovarianzmatrix des Datensatzes.

Zur Verdeutlichung berechnen wir die Kovarianzmatrix der 3D-Daten und führen anschließend eine Singulärwertzerlegung (SVD) durch. Diese liefert uns:

- **Eigenvektoren**: Die Richtungen der Hauptachsen im Raum,
- **Eigenwerte**: Die Streuung (Varianz) entlang dieser Hauptachsen.


In [ ]:
kovmat_3d = np.cov(X_3d.T)
u_3d,s_3d,v_3d = np.linalg.svd(kovmat_3d)
print("Eigenvektoren :", v_3d)
print("Eigenwerte:", s_3d)

### Visualisierung der PCA-Richtungen im 3D-Raum

Um ein besseres Gefühl für die Funktionsweise der PCA zu bekommen, visualisieren wir die 3D-Daten gemeinsam mit den Hauptkomponenten.

Die Hauptachsen (bzw. -richtungen) werden als Pfeile dargestellt, die vom Mittelwert der Daten ausgehen. Dabei gilt:

- Die Richtung jedes Pfeils entspricht der Richtung einer Hauptkomponente (Eigenvektor),
- Die Länge ist proportional zur erklärten Varianz (also der Wurzel des Eigenwerts),
- Die Farben kennzeichnen die drei Komponenten (rot, blau, grün).


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], alpha=0.2)
origin = np.mean(X_3d, axis=0)  # Besser: Mittelwert der Daten als Ursprung


ursprung_3d = (0,0)
pc_3d = []
colors = ['r', 'b', 'g']
for i in range(len(v_3d)):
    vec = v_3d[i] * np.sqrt(s_3d[i])
    ax.quiver(*origin, *vec, color=colors[i], linewidth=2)

# Achsen beschriften
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title("3D-Daten mit PCA-Richtungen")
plt.tight_layout()

#plt.savefig("pca_3d_pca.png", dpi=300)
#plt.savefig("pca_3d_pca.svg")

plt.show()

### Projektion der 3D-Daten auf 2 Hauptkomponenten

Nachdem wir den Datensatz in 3D generiert und die Hauptkomponenten berechnet haben, wollen wir nun die Daten auf nur **zwei Hauptkomponenten reduzieren**.

Mit der Methode `transform()` von *Scikit-Learn* wird der Datensatz auf die ersten beiden Hauptkomponenten projiziert. Dadurch erhalten wir eine Darstellung der Daten mit reduzierter Dimension – von ursprünglich 3 auf 2.


In [ ]:
pca = PCA(n_components=2)
pca.fit(X_3d)
X_pca = pca.transform(X_3d)
print("Ursprüngliche Größe:   ", X_3d.shape)
print("UTransformierte Größe:", X_pca.shape)
print("Der Datensatz enthält %.1f%c der Varianz der ersten Komponente." % (100 * pca.explained_variance_ratio_[0], '%'))
print("Der Datensatz enthält %.1f%c der Varianz der zweiten Komponente." % (100 * pca.explained_variance_ratio_[1], '%'))


## PCA-Projektion mit 1σ- und 3σ-Ellipsen

Der Plot zeigt die Projektion der Daten auf die ersten beiden Hauptkomponenten (PC1, PC2) der PCA. Die Pfeile markieren die Richtungen der Hauptkomponenten mit einer Länge von $3\sigma$, wobei $\sigma$ die Standardabweichung entlang der jeweiligen PC ist.

Die $1\sigma$-Ellipse (gestrichelt) entspricht dem Bereich, in dem ca. 68 % der Datenpunkte bei normalverteilten Daten liegen. Die $3\sigma$-Ellipse (punktiert) umfasst etwa 99,7 % der Datenpunkte.

Diese Ellipsen veranschaulichen die Streuung und Verteilung der Daten im Raum der Hauptkomponenten.


In [ ]:
# PCA-Variablen für Plot
pcs = X_pca
std_pcs = np.sqrt(pca.explained_variance_)

fig, ax = plt.subplots(figsize=(8, 8))

# Scatterplot der PCA-Daten
ax.scatter(pcs[:, 0], pcs[:, 1], alpha=0.6, c='blue')

# Pfeile: PC1 horizontal nach rechts, PC2 vertikal nach oben
ax.plot([0, 3 * std_pcs[0]], [0, 0], color='C1', lw=3, label="Richtung PC 1, Länge = 3σ")
ax.plot([0, 0], [0, 3 * std_pcs[1]], color='C3', lw=3, label="Richtung PC 2, Länge = 3σ")

# 1σ Ellipse
ellipse_1sigma = Ellipse(xy=(0, 0),
                        width=2 * std_pcs[0], height=2 * std_pcs[1],
                        edgecolor='black', fc='None', ls='--', lw=2,
                        label=r'1$\sigma$ Ellipse')
ax.add_patch(ellipse_1sigma)

# 3σ Ellipse
ellipse_3sigma = Ellipse(xy=(0, 0),
                        width=6 * std_pcs[0], height=6 * std_pcs[1],
                        edgecolor='black', fc='None', ls=':', lw=2,
                        label=r'3$\sigma$ Ellipse')
ax.add_patch(ellipse_3sigma)

plt.xlabel("1. Hauptkomponente")
plt.ylabel("2. Hauptkomponente")
ax.set_title("2D Projektion mit 1σ und 3σ Ellipsen")
ax.legend()
ax.grid(True)
ax.axis('equal')

#plt.savefig("pca_3d_ell.png", dpi=300)
#plt.savefig("pca_3d_ell.svg")

plt.show()

### Visualisierung der 2D-Projektion

Nach der Reduktion des 3D-Datensatzes auf zwei Hauptkomponenten können wir den projizierten Datensatz in einem 2D-Streudiagramm darstellen.

Das Ergebnis ist eine Visualisierung der Daten **in einem neuen Koordinatensystem**, das durch die beiden wichtigsten Richtungen der Varianz (Hauptkomponenten) definiert ist.


In [ ]:
# Visualisiere 2D-Projektion (Reduktion von 3D auf 2D)
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, c='blue')
plt.xlabel("1. Hauptkomponente")
plt.ylabel("2. Hauptkomponente")
plt.title("2D-Projektion der 3D-Daten durch PCA")
plt.axis('equal')

#plt.savefig("pca_3d_proj.png", dpi=300)
#plt.savefig("pca_3d_proj.svg")

plt.show()

### Rekonstruktion im 3D-Raum

Die Hauptkomponentenanalyse ermöglicht nicht nur die Projektion in niedrigere Dimensionen, sondern auch die **Rekonstruktion** der Daten aus den reduzierten Komponenten.

Dabei wird versucht, die ursprünglichen Daten aus der 2D-Projektion im 3D-Raum **näherungsweise wiederherzustellen**

In [ ]:
# Rekonstruktion und Visualisierung im 3D-Raum
X_reconstructed = pca.inverse_transform(X_pca)

### Visualisierung der Rekonstruktion im 3D-Raum

Wir visualisieren nun den ursprünglichen 3D-Datensatz zusammen mit den rekonstruierten Daten, die aus der 2D-PCA-Projektion zurücktransformiert wurden.

- Die **Originaldaten** sind transparent dargestellt, damit die Verteilung sichtbar bleibt.
- Die **rekonstruierten Punkte** liegen auf der Ebene, die durch die zwei Hauptkomponenten aufgespannt wird.
- Die orangefarbene Fläche zeigt genau diese PCA-Ebene.


In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')

# Original- und rekonstruierte Daten
ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], alpha=0.1, label="Originaldaten")
ax.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], X_reconstructed[:, 2],
           alpha=0.8, label="Rekonstruiert (2D)", c='b')

# PCA-Ebene zeichnen
origin = pca.mean_
vec1, vec2 = pca.components_
range1 = np.linspace(-3, 3, 20)
range2 = np.linspace(-3, 3, 20)
xx, yy = np.meshgrid(range1, range2)
plane = origin + xx[..., np.newaxis]*vec1 + yy[..., np.newaxis]*vec2
ax.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], alpha=0.3, color='orange')

# Achsenbeschriftung
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_zlabel(r"$z$")
plt.title("Rekonstruktion aus PCA (3D)")
ax.legend()

#plt.savefig("pca_3d_hyp.png", dpi=300)
#plt.savefig("pca_3d_hyp.svg")

plt.show()

print("Der Datensatz enthält %.1f%% der ursprünglichen Information (Varianz)." % 
      (100 * np.sum(pca.explained_variance_ratio_)))

### Interaktive Visualisierung der PCA-Rekonstruktion im 3D-Raum

Zur besseren Veranschaulichung nutzen wir nun **Plotly**, um den ursprünglichen 3D-Datensatz, die rekonstruierten Daten aus der 2D-PCA und die PCA-Ebene interaktiv darzustellen.

- Die **Originaldaten** sind als grüne, transparente Punkte dargestellt.
- Die **rekonstruierten Daten** erscheinen in kräftigem Blau und liegen auf der orangefarbenen PCA-Ebene.
- Die PCA-Ebene wird als transparente Fläche visualisiert und repräsentiert den Unterraum, auf den die Daten bei der Dimensionsreduktion projiziert wurden.

Durch interaktive Drehung und Zoom kann man den Raum aus verschiedenen Blickwinkeln betrachten und so die Dimensionreduktion besser verstehen.


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Mittelwert und PCA-Komponenten
origin = pca.mean_
vec1, vec2 = pca.components_

# Gitter für die PCA-Ebene
range1 = np.linspace(-3, 3, 20)
range2 = np.linspace(-3, 3, 20)
xx, yy = np.meshgrid(range1, range2)
plane = origin + xx[..., np.newaxis]*vec1 + yy[..., np.newaxis]*vec2

# 3D Scatter Originaldaten
scatter_orig = go.Scatter3d(
    x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
    mode='markers',
    marker=dict(size=3, color='green', opacity=0.1),
    name='Originaldaten'
)

# 3D Scatter Rekonstruierte Daten
scatter_recon = go.Scatter3d(
    x=X_reconstructed[:, 0], y=X_reconstructed[:, 1], z=X_reconstructed[:, 2],
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.8),
    name='Rekonstruiert (2D)'
)

# PCA-Ebene als Mesh3d (Fläche)
plane_surface = go.Mesh3d(
    x=plane[..., 0].flatten(),
    y=plane[..., 1].flatten(),
    z=plane[..., 2].flatten(),
    alphahull=0,
    opacity=0.3,
    color='orange',
    name='PCA Ebene'
)

# Layout mit Achsenbeschriftungen und Titel
layout = go.Layout(
    width=800, 
    height=600, 
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
    ),
    title='Rekonstruktion aus PCA (3D)'
)

# Figure zusammenbauen und anzeigen
fig = go.Figure(data=[scatter_orig, scatter_recon, plane_surface], layout=layout)

#pio.write_image(fig, "3d_plot.png")
#pio.write_image(fig, "3d_plot.svg")

fig.show()


## Principal Component Analysis mit Audio-Merkmalsdaten

In diesem Abschnitt betrachten wir einen Datensatz, der aus vorverarbeiteten Audiodaten besteht. Dabei handelt es sich **nicht um echte Audiosignale oder Audiodateien**, sondern um **numerische Merkmale**, die zuvor aus solchen Signalen extrahiert wurden. Die Daten liegen im komprimierten Format `.npz` vor und wurden bereits für weitere Analysen vorbereitet.

Die Datei `_raw_data_large.npz` enthält zwei Arrays:

- `Xdata`: ein 2D-Array mit extrahierten Merkmalen pro Audiobeispiel.
- `Ydata`: zugehörige Zielwerte (z. B. für Klassifikation oder Regression).

Die Spalten von `Xdata` enthalten folgende Merkmale:

| Index | Merkmal         | Beschreibung                                      |
|-------|------------------|---------------------------------------------------|
| 0     | true_peak_lin    | Linearer Spitzenpegel                             |
| 1     | true_peak_lin2   | Alternative Berechnung des Spitzenpegels         |
| 2     | true_peak_db     | Spitzenpegel in Dezibel                          |
| 3     | rms_lin2         | Zweite Variante des RMS-Werts (Effektivwert)     |
| 4     | rms_lin          | RMS-Wert (Effektivwert), linear                  |
| 5     | rms_db           | RMS-Wert in Dezibel                             |
| 6     | lufs_lin         | Lautheit (LUFS), linear                          |
| 7     | lufs_lin2        | Alternative LUFS-Berechnung, linear              |
| 8     | lufs_db          | Lautheit (LUFS) in Dezibel                      |
| 9     | crest_lin        | Crest-Faktor (Spitzenwert/RMS), linear           |
| 10    | crest_db         | Crest-Faktor in Dezibel                          |
| 11    | low_high_ratio   | Verhältnis von Tiefen- zu Höhenanteil im Signal  |

Die Zielwerte `Ydata` repräsentieren Klassenzugehörigkeiten der Audiodaten und sind wie folgt kodiert:

| Label | Genre     |
|-------|------------|
| 0     | Metal      |
| 1     | EDM        |
| 2     | Classical  |


### Scatter-Matrix zur visuellen Analyse der Merkmale

Bevor wir eine PCA (Principal Component Analysis) durchführen oder einzelne Merkmale auswählen, möchten wir uns einen Überblick über alle verfügbaren Features verschaffen. Dazu nutzen wir eine sogenannte **Scatter-Matrix** (auch: Streudiagramm-Matrix).

In der Scatter-Matrix wird **jedes Feature gegen jedes andere geplottet**. So können wir visuell erkennen:

- welche Merkmale **stark korreliert** sind (z. B. durch eine diagonale Struktur),
- welche Merkmale **unterschiedliche Verteilungen** aufweisen,
- und welche **relevanten Trennungen oder Cluster** evtl. schon sichtbar sind.

Diese Visualisierung hilft uns dabei, erste Hypothesen zu bilden, **welche Features besonders informativ** sein könnten – z. B. weil sie mit anderen stark zusammenhängen oder bestimmte Muster zeigen. 

Ziel: Wir möchten auf Basis dieser Matrix abschätzen, **welche Features potenziell redundant** oder besonders **aussagekräftig für spätere Analysen** sein könnten.


In [ ]:
import pandas as pd
from pandas.plotting import scatter_matrix

audiofolder = "./audio_data/"
with np.load(audiofolder + "/_raw_data_large.npz") as data:
    X = data["Xdata"]
    Y = data["Ydata"]


X_raw = X.copy()  # oder: X_full = X.copy()

# Feature-Namen laut deiner Tabelle
feature_names = [
    "true_peak_lin", "true_peak_lin2", "true_peak_db",
    "rms_lin2", "rms_lin", "rms_db",
    "lufs_lin", "lufs_lin2", "lufs_db",
    "crest_lin", "crest_db", "low_high_ratio"
]

# Erstelle vollständigen DataFrame mit allen Features
df_all = pd.DataFrame(X, columns=feature_names)

# Scatter-Matrix über alle Features
scatter_matrix(df_all, figsize=(14, 14), alpha=0.5)
plt.suptitle("Scatter Matrix aller 12 Features", y=1.02)

#plt.savefig("audio_scatter.png", dpi=300)
#plt.savefig("audio_scatter.svg")

plt.show()

### Kumulative erklärte Varianz

Mit der Principal Component Analysis (PCA) können wir herausfinden, **wie viele Hauptkomponenten** nötig sind, um möglichst viel Information (Varianz) aus unseren Daten zu bewahren.

In diesem Plot sehen wir, **wie stark sich die erklärte Varianz summiert**, wenn wir mehr Hauptkomponenten hinzufügen. Ziel ist es, mit möglichst wenigen Komponenten einen möglichst großen Anteil der Gesamtvarianz zu erfassen.

**Beobachtung:**  
Ab einer bestimmten Anzahl an Komponenten flacht die Kurve ab – zusätzliche Komponenten bringen dann nur noch wenig zusätzlichen Informationsgewinn.

---

> Hinweis:  
Unabhängig davon, wie viele Komponenten mathematisch sinnvoll wären, beschränken wir uns in dieser Übung bewusst auf **3 Hauptkomponenten**, da wir die Daten später **dreidimensional visualisieren** wollen. Dadurch bleibt die Analyse für uns **anschaulich und leicht interpretierbar**.


In [ ]:
X_full = np.copy(X_raw)  

# Zentrieren und normalisieren (z-Score-Normalisierung)
X_full = X_full - np.mean(X_full, axis=0)
X_full = X_full / np.std(X_full, axis=0, ddof=1)

# PCA durchführen
pca = PCA()
pca.fit(X_full)

# Kumulative erklärte Varianz berechnen
exp_var_ratio = pca.explained_variance_ratio_
exp_var_cumul = np.cumsum(exp_var_ratio)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(exp_var_cumul) + 1), exp_var_cumul, marker="o", linestyle="-")
plt.xticks(range(1, len(exp_var_cumul) + 1))
plt.xlabel("Anzahl der Hauptkomponenten")
plt.ylabel("Kumulative erklärte Varianz")
plt.title("PCA: Kumulative erklärte Varianz (alle Features)")
plt.grid(True)
plt.tight_layout()

#plt.savefig("pca_cum_var_all.png", dpi=300)
#plt.savefig("pca_cum_var_all.svg")

plt.show()


Wir definieren hier unseren Datensatz neu und beschränken uns bewusst auf 3 ausgewählte Features, um die Daten später sinnvoll in 3D visualisieren zu können. Später führen wir eine PCA durch, um die Daten für eine anschauliche 2D-Darstellung zu reduzieren.

Die Auswahl der Merkmale ist nicht fest vorgegeben — es können auch ```andere Features``` aus der Liste verwendet werden. Je nach Analysezweck und Fragestellung lohnt es sich, mit unterschiedlichen Kombinationen der Merkmale zu ```experimentieren```, um herauszufinden, welche Features für die Hauptkomponentenanalyse (PCA) die besten Ergebnisse liefern.

In [ ]:
# Wähle bestimmte Features: 
selected_features = [3, 7, 10]
X = np.squeeze([X[:, i] for i in selected_features]).T

# Zentriere die Daten: Ziehe von jeder Spalte den Mittelwert ab
X = X - np.mean(X, axis=0)

# Skaliere jede Spalte auf Standardabweichung = 1 (z-Score Normalisierung)
X = X / np.std(X, axis=0, ddof=1)

# Speichere Form der Datenmatrix
N, F = X.shape[0], X.shape[1]
N, F


### 3D-Visualisierung der Original-Audio-Merkmale

Dieser 3D-Scatterplot zeigt die Verteilung der Audiodaten im ursprünglichen Merkmalsraum mit den drei ausgewählten Features:

- `3: true_peak_lin` (x-Achse)
- `7: lufs_lin2` (y-Achse)
- `10: crest_db` (z-Achse)

Die Punkte bilden eine Datenwolke im dreidimensionalen Raum, die wir später durch PCA in eine niedrigere Dimension transformieren möchten.  
Die Kameraperspektive wurde so gewählt, dass man die Verteilung gut erkennen kann.


In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

# Scatter-Plot mit Farbcodierung nach Y
sc = ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=Y, cmap="viridis", s=10)

# Colorbar 
cb = fig.colorbar(sc, ax=ax, pad=0.15, ticks=[0.0, 1.0, 2.0], shrink=0.6)
cb.set_label("Label")

# Graue Punkte als Zusatzvisualisierung
ax.plot(X[:, 0], X[:, 1], X[:, 2], ".", color="gray", ms=1)

# Achsenbeschriftungen mit Feature-Namen
ax.set_xlabel(feature_names[selected_features[0]])
ax.set_ylabel(feature_names[selected_features[1]])
ax.set_zlabel(feature_names[selected_features[2]])

ax.set_title("Datenwolke im originalen Koordinatensystem")
ax.grid(True)

plt.tight_layout()

#plt.savefig("audio_data.png", dpi=300)
#plt.savefig("audio_data.svg")

plt.show()


### Hauptkomponentenanalyse (PCA) mit drei Komponenten

In diesem Abschnitt führen wir eine PCA mit drei Hauptkomponenten auf den zuvor ausgewählten und normalisierten Audiodaten durch. 

- Zunächst wird das PCA-Modell mit `n_components=3` erstellt und auf die Daten `X` angepasst.
- Anschließend transformieren wir die Daten in den neuen dreidimensionalen Hauptkomponentenraum.
- Die Ausgabe zeigt die Dimensionen der ursprünglichen und transformierten Daten.
- Zudem geben wir den Anteil der Varianz an, den jede der drei Hauptkomponenten erklärt.

Die Varianzanteile zeigen, wie gut die Hauptkomponenten die Daten in reduzierter Dimension repräsentieren. Je höher der Anteil, desto mehr Information wird durch diese Komponente erfasst.


In [ ]:
pca_audio_3d = PCA(n_components=3)
pca_audio_3d.fit(X)
X_audio_3d = pca_audio_3d.transform(X)
print("Ursprüngliche Größe:   ", X.shape)
print("UTransformierte Größe:", X_audio_3d.shape)
print("Der Datensatz enthält %.1f%c der Varianz der ersten Komponente." % (100 * pca_audio_3d.explained_variance_ratio_[0], '%'))
print("Der Datensatz enthält %.1f%c der Varianz der zweiten Komponente." % (100 * pca_audio_3d.explained_variance_ratio_[1], '%'))
print("Der Datensatz enthält %.1f%c der Varianz der zweiten Komponente." % (100 * pca_audio_3d.explained_variance_ratio_[2], '%'))

Wir geben die Hauptkomponenten (Eigenvektoren) der PCA und die zugehörigen Eigenwerte (explained variance) aus. Die Komponenten zeigen die Richtungen im Merkmalsraum, auf die die Daten am stärksten variieren, während die Eigenwerte den Betrag der erklärten Varianz in diesen Richtungen angeben.

In [ ]:
print(pca_audio_3d.components_)
print(pca_audio_3d.explained_variance_)

Zur Validierung der PCA-Ergebnisse berechnen wir die Kovarianzmatrix der transformierten Daten und wenden die Singulärwertzerlegung (SVD) an. 

Die Matrix `v_audio_3d` enthält die Eigenvektoren, und `s_audio_3d` die zugehörigen Eigenwerte der Kovarianzmatrix. Diese stimmen mit den PCA-Komponenten und deren Varianzen überein und bestätigen damit die Konsistenz der PCA-Transformation.


In [ ]:
kovmat_audio_3d = np.cov(X_audio_3d.T)
u_audio_3d,s_audio_3d,v_audio_3d = np.linalg.svd(kovmat_audio_3d)
print("Eigenvektoren :", v_audio_3d)
print("Eigenwerte:", s_audio_3d)

In dieser 3D-Visualisierung werden die standardisierten Audiodaten als graue Punkte dargestellt. Der Ursprung ist dabei der Mittelwert der Daten, um eine bessere Zentrierung zu gewährleisten.

Die farbigen Pfeile repräsentieren die Hauptkomponenten (PCA-Richtungen), welche durch die Eigenvektoren der Kovarianzmatrix gewichtet mit den Wurzeln der Eigenwerte dargestellt werden. Diese Richtungen zeigen die Hauptachsen, entlang derer die Varianz der Daten maximal ist.

- Rot: Erste Hauptkomponente (größte Varianz)
- Blau: Zweite Hauptkomponente
- Grün: Dritte Hauptkomponente

Diese Visualisierung verdeutlicht, wie die PCA die ursprünglichen Merkmale in ein neues Koordinatensystem überführt, das die wichtigsten Varianzrichtungen abbildet.


In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(X[:, 0], X[:, 1], X[:, 2], ".", color="gray", ms=1)

origin_audio = np.mean(X, axis=0)  # Besser: Mittelwert der Daten als Ursprung


ursprung_audio = (0,0)
pc_3d = []
colors = ['r', 'b', 'g']
for i in range(len(v_audio_3d)):
    vec_audio = v_audio_3d[i] * np.sqrt(s_audio_3d[i])
    ax.quiver(*origin_audio, *vec_audio, color=colors[i], linewidth=2)

# Achsen beschriften
ax.set_xlabel(feature_names[selected_features[0]])
ax.set_ylabel(feature_names[selected_features[1]])
ax.set_zlabel(feature_names[selected_features[2]])
ax.set_title("Audio-Daten mit PCA-Richtungen")
plt.tight_layout()

#plt.savefig("pca_audio_dir.png", dpi=300)
#plt.savefig("pca_audio_dir.svg")

plt.show()

Hier analysieren wir die kumulative erklärte Varianz für die zuvor ausgewählten 3 Features. Der Plot zeigt, wie viel Information (Varianz) durch die PCA-Hauptkomponenten jeweils erklärt wird. Da wir nur 3 Features verwenden, gibt es auch nur 3 Hauptkomponenten – entsprechend endet die Kurve bei 100 %. So können wir überprüfen, wie stark bereits die ersten ein oder zwei Komponenten zur Gesamtvarianz beitragen.

In [ ]:
X_full = np.copy(X)

# Zentrieren und normalisieren (z-Score-Normalisierung)
X_full = X_full - np.mean(X_full, axis=0)
X_full = X_full / np.std(X_full, axis=0, ddof=1)

# PCA durchführen
pca = PCA()
pca.fit(X_full)

# Kumulative erklärte Varianz berechnen
exp_var_ratio = pca.explained_variance_ratio_
exp_var_cumul = np.cumsum(exp_var_ratio)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(exp_var_cumul) + 1), exp_var_cumul, marker="o", linestyle="-")
plt.xticks(range(1, len(exp_var_cumul) + 1))
plt.xlabel("Anzahl der Hauptkomponenten")
plt.ylabel("Kumulative erklärte Varianz")
plt.title("PCA: Kumulative erklärte Varianz (3 Features)")
plt.grid(True)
plt.tight_layout()

#plt.savefig("pca_cum_var.png", dpi=300)
#plt.savefig("pca_cum_var.svg")

plt.show()


Wir führen nun eine Hauptkomponentenanalyse (PCA) mit zwei Komponenten durch,  um die ursprünglichen Daten von 3 auf 2 Dimensionen zu reduzieren.


- Die ursprünglichen Daten besitzen die Form `(N, F)` mit `N` Datenpunkten und `F=3` Merkmalen.
- Nach der Transformation haben die Daten die Form `(N, 2)`, da nur die zwei wichtigsten Hauptkomponenten beibehalten werden.

Die erste Hauptkomponente erklärt dabei `84.3%` der Gesamtvarianz, die zweite Hauptkomponente `12.5%`. Zusammen erfassen diese zwei Komponenten somit den größten Teil der Varianz der ursprünglichen Daten, was eine gute Reduktion der Dimensionalität erlaubt.


In [ ]:
pca_audio = PCA(n_components=2)
pca_audio.fit(X)
X_pca_audio = pca_audio.transform(X)
print("Ursprüngliche Größe:   ", X.shape)
print("UTransformierte Größe:", X_pca_audio.shape)
print("Der Datensatz enthält %.1f%c der Varianz der ersten Komponente." % (100 * pca_audio.explained_variance_ratio_[0], '%'))
print("Der Datensatz enthält %.1f%c der Varianz der zweiten Komponente." % (100 * pca_audio.explained_variance_ratio_[1], '%'))


Wir visualisieren die Projektion der Daten auf die ersten zwei Hauptkomponenten. Jeder Punkt entspricht einem Datenbeispiel, eingefärbt nach seinem Label (Genre).

Die x-Achse zeigt die erste Hauptkomponente (PC1), die y-Achse die zweite Hauptkomponente (PC2).

In [ ]:
plt.figure(figsize=(8,8))
sc = plt.scatter(X_pca_audio[:, 0], X_pca_audio[:, 1], c=Y, cmap="viridis", s=10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projektion")

cb = plt.colorbar(sc, pad=0.15, ticks=[0, 1, 2], shrink=0.6)
cb.set_label("Label")

#plt.savefig("pca_audio.png", dpi=300)
#plt.savefig("pca_audio.svg")

plt.grid(True)
plt.show()


Wir rekonstruieren die Daten aus der zweidimensionalen PCA-Projektion zurück in den ursprünglichen dreidimensionalen Merkmalsraum. Dies ermöglicht eine Visualisierung, wie gut die reduzierte Darstellung die Originaldaten approximiert. Die Rekonstruktion zeigt, wie viel Information bei der Dimensionsreduktion erhalten bleibt.


In [ ]:
# Rekonstruktion und Visualisierung im 3D-Raum
X_reconstructed_audio = pca_audio.inverse_transform(X_pca_audio)

Wir visualisieren die ursprünglichen Audiodaten zusammen mit den aus der zweidimensionalen PCA-Projektion rekonstruierten Daten im dreidimensionalen Merkmalsraum. Die Originaldaten sind grau und transparent dargestellt, während die rekonstruierten Punkte blau hervorgehoben sind.

Außerdem wird die durch die zwei Hauptkomponenten definierte Ebene als orangefarbene Fläche eingezeichnet. Diese Ebene repräsentiert den reduzierten Raum, auf den die Daten projiziert wurden.

Die Ausgabe zeigt den Anteil der ursprünglichen Varianz, der durch die zwei Komponenten erhalten bleibt, was die Qualität der Dimensionsreduktion verdeutlicht.


In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Original- und rekonstruierte Daten
ax.scatter(X[:, 0], X[:, 1], X[:, 2], alpha=0.1, color='gray', label="Originalaudiodaten")
ax.scatter(X_reconstructed_audio[:, 0], X_reconstructed_audio[:, 1], X_reconstructed_audio[:, 2],
           alpha=0.9, color='blue', label="Rekonstruiert (2D)")

# PCA-Ebene zeichnen
origin = pca_audio.mean_
vec1, vec2 = pca_audio.components_
range1 = np.linspace(-3, 3, 20)
range2 = np.linspace(-3, 3, 20)
xx, yy = np.meshgrid(range1, range2)
plane = origin + xx[..., np.newaxis]*vec1 + yy[..., np.newaxis]*vec2
ax.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], alpha=0.3, color='orange')

# Achsenbeschriftung
ax.set_xlabel(feature_names[selected_features[0]])
ax.set_ylabel(feature_names[selected_features[1]])
ax.set_zlabel(feature_names[selected_features[2]])
plt.title("Rekonstruktion aus PCA (Audio)")
ax.legend()


#plt.savefig("pca_rekonstruktion_audio.png", dpi=300)
#plt.savefig("pca_rekonstruktion_audio.svg")

ax.set_box_aspect(None, zoom=0.85)
plt.show()

print("Der Datensatz enthält %.1f%% der ursprünglichen Information (Varianz)." % 
      (100 * np.sum(pca_audio.explained_variance_ratio_)))


### Visualisierung der PCA-Rekonstruktion im 3D-Raum

In dieser Darstellung sehen wir die Originaldaten als grüne, sehr transparente Punkte und die aus der zweidimensionalen PCA-Projektion rekonstruierten Daten als kräftig blaue Punkte. Zusätzlich wird die von den ersten zwei Hauptkomponenten aufgespannte Ebene (PCA-Ebene) orange als Fläche angezeigt.

Die Nähe der rekonstruierten Punkte zur PCA-Ebene zeigt, wie gut die zweidimensionale Projektion die ursprünglichen Daten approximiert. Für die hier ausgewählten Features scheint es, als lägen die Originaldaten nahezu auf dieser Ebene. Tatsächlich befinden sich die `Originaldaten jedoch knapp ober- oder unterhalb der Ebene`, was bei anderen Feature-Kombinationen oft noch deutlicher wird.

Diese Visualisierung verdeutlicht den Kompromiss bei der Dimensionsreduktion: Wir reduzieren von drei auf zwei Dimensionen und erfassen dabei einen Großteil der Varianz, verlieren aber gleichzeitig einen kleinen Teil an Information, der sich in der Entfernung der Originalpunkte von der PCA-Ebene zeigt.


In [ ]:
# Mittelwert und PCA-Komponenten
origin = pca_audio.mean_
vec1, vec2 = pca_audio.components_

# Gitter für die PCA-Ebene
range1 = np.linspace(-3, 3, 20)
range2 = np.linspace(-3, 3, 20)
xx, yy = np.meshgrid(range1, range2)
plane = origin + xx[..., np.newaxis]*vec1 + yy[..., np.newaxis]*vec2

# 3D Scatter Originaldaten
scatter_orig = go.Scatter3d(
    x=X[:, 0], y=X[:, 1], z=X[:, 2],
    mode='markers',
    marker=dict(size=3, color='green', opacity=0.1),
    name='Originaldaten'
)

# 3D Scatter Rekonstruierte Daten
scatter_recon = go.Scatter3d(
    x=X_reconstructed_audio[:, 0], y=X_reconstructed_audio[:, 1], z=X_reconstructed_audio[:, 2],
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.8),
    name='Rekonstruiert (2D)'
)

# PCA-Ebene als Mesh3d (Fläche)
plane_surface = go.Mesh3d(
    x=plane[..., 0].flatten(),
    y=plane[..., 1].flatten(),
    z=plane[..., 2].flatten(),
    alphahull=0,
    opacity=0.3,
    color='orange',
    name='PCA Ebene'
)

# Layout mit Achsenbeschriftungen und Titel
layout = go.Layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
    ),
    title='Rekonstruktion aus PCA (Audio)'
)

# Figure zusammenbauen und anzeigen
fig = go.Figure(data=[scatter_orig, scatter_recon, plane_surface], layout=layout)

#pio.write_image(fig, "3d_plot.png")
#pio.write_image(fig, "3d_plot.svg")

fig.show()


### Vergleich: Logistische Regression auf Originaldaten vs. PCA-Daten

In diesem Beispiel wird die logistische Regression auf den originalen 3D-Daten und auf den durch PCA auf 2D reduzierten Daten angewendet. Ziel ist es, die Rechenzeit zu vergleichen und die Auswirkung der Dimensionsreduktion auf die Genauigkeit zu beurteilen.

Durch Anwendung von PCA reduzieren wir die ursprünglichen 3D-Daten auf zwei Hauptkomponenten. Die Accuracy bleibt in der Regel ähnlich, da PCA die wichtigsten Strukturen erhält, während die Rechenzeit dank weniger Dimensionen meist sinkt.

---

Hier verwenden wir [%timeit](https://docs.python.org/3/library/timeit.html), um die Ausführungszeit der cross-validierten logistischen Regression zu messen.

- `-n`: Anzahl der Wiederholungen pro Lauf (hier `-n1` bedeutet 1 Wiederholung pro Lauf)
- `-r`: Anzahl der kompletten Läufe (hier `-r1` bedeutet, dass nur 1 Lauf durchgeführt wird)


In [ ]:
# Standardisieren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf_lr = LogisticRegression(max_iter=1000)

# Zeit + Accuracy auf Originaldaten (3D)
print("Logistische Regression auf Originaldaten (3D):")
%timeit -n1 -r1 cross_val_score(clf_lr, X_scaled, Y.ravel(), cv=5)
scores_orig = cross_val_score(clf_lr, X_scaled, Y.ravel(), cv=5)
print(f"Accuracy (3D): {np.mean(scores_orig):.2f}")

# PCA auf 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

clf_lr_pca = LogisticRegression(max_iter=1000)

print("\nLogistische Regression auf PCA-Daten (2D):")
%timeit -n1 -r1 cross_val_score(clf_lr_pca, X_pca, Y.ravel(), cv=5)
scores_pca = cross_val_score(clf_lr_pca, X_pca, Y.ravel(), cv=5)
print(f"Accuracy (PCA-2D): {np.mean(scores_pca):.2f}")


Die logistische Regression auf den **Originaldaten (3D)** erreicht eine **Accuracy von 0.81** bei einer Rechenzeit von **715 ms**.  
Nach der **Dimensionsreduktion mit PCA auf 2D** sinkt die Accuracy leicht auf **0.74**, dafür reduziert sich die Rechenzeit deutlich auf **346 ms**.

**Fazit:**  
PCA führt zu einem spürbaren Zeitgewinn bei nur geringem Genauigkeitsverlust. Für schnelle oder visuelle Analysen kann PCA daher sinnvoll sein – insbesondere bei größeren Datensätzen.

In [ ]:
from matplotlib.colors import ListedColormap

clf_lr_pca.fit(X_pca, Y.ravel())

h = 0.02
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

Z = clf_lr_pca.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

cmap_light = ListedColormap(["#FFAAAA", "#AAAAFF", "#AAFFAA"])
cmap_bold = ListedColormap(["#FF0000", "#0000FF", "#00FF00"])

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.4)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=Y.ravel(), cmap=cmap_bold, edgecolor="k", s=50)

plt.xlabel("PCA-Komponente 1")
plt.ylabel("PCA-Komponente 2")
plt.title("Logistische Regression: Entscheidungsgrenze auf PCA (2D)")
plt.legend(*scatter.legend_elements(), title="Klasse")
plt.grid(True)
plt.tight_layout()

#plt.savefig("pca_audio_log.png", dpi=300)
#plt.savefig("pca_audio_log.svg")

plt.show()


### PCA zur Visualisierung

Das Einsatzspektrum von PCA zeigt sich besonders bei Datensätzen mit sehr hoher Dimensionen.
Im Folgenden wollen wir die PCA-Methode auf einen [Bild-Datensatz (genauer dem UCI ML hand-written digits dataset)](https://archive.ics.uci.edu/ml/datasets/Optical+Recognition+of+Handwritten+Digits) mit weit mehr als $2$ Merkmalen anwendender mittels der [`load_digits()`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) Funktion geladen werden kann:

In [ ]:
digits = load_digits()
print(f"Ziffern Datensatz mit {digits.data.shape[0]} Datenpunkten und {digits.data.shape[1]} Merkmalen")

_, axes = plt.subplots(nrows=1, ncols=4, figsize=(10, 3))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.set_axis_off()
    ax.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
    ax.set_title('Training: %i' % label)
    
#plt.savefig("pca_digit.png", dpi=300)
#plt.savefig("pca_digit.svg")

plt.show()

Jeder Datenpunkt besteht aus einem grob aufgelösten Graustufenbild von $8 \times 8$ Pixeln einer handgeschriebenen Ziffer.
Jedes Pixel wird als Merkmal gedeutet.
In Summe haben wir es hier also mit $64$ unterschiedlichen Variablen zu tun.

#### Beispiel: Dimensionsreduktion von 64 auf 2 Dimensionen mittels PCA

Wir wollen nun den Datensatz auf zwei Hauptkomponenten reduzieren:

In [ ]:
pca = PCA(2)  # project from 64 to 2 dimensions
projected = pca.fit_transform(digits.data)
print(digits.data.shape)
print(projected.shape)
print("Der Datensatz enthält noch %.1f%c der ursprünglichen Information (Varianz)"
      % (100*sum(pca.explained_variance_ratio_), '%'))

Da wir nun unsere Ziffern-Bilder durch 2 Merkmale beschreiben, können wir die Datenpunkte in der Ebene plotten:

In [ ]:
plt.scatter(projected[:, 0], projected[:, 1],
            c=digits.target, edgecolor='none', alpha=0.5,
            cmap=plt.get_cmap('viridis', 10))
plt.xlabel('component 1')
plt.ylabel('component 2')
plt.colorbar()

#plt.savefig("pca_digit_color", dpi=300)
#plt.savefig("pca_digit_color.svg")

plt.show()

In [ ]:
pca.explained_variance_ratio_[0], pca.explained_variance_ratio_[1]

Was bedeutet diese Vereinfachung?

Wir sind nun von $64$ Pixeln auf $2$ Merkmale herunter gekommen, haben unseren Datensatz also um den Faktor $32$ verkleinert.
Damit erhalten wir nicht nur weniger Daten, auf eine Modellfunktion lässt sich auf weniger Merkmale hin deutlich leichter trainieren.

Allerdings müssen wir uns die Frage stellen, ob $2$ Merkmale hier noch ausreichend für gute Vorhersagen sind.
Wir sehen zwar in der Abbildung, dass die Punktwolken für die einzelnen Ziffern gebündelt auftreten.
Aber die Wolken reichen doch deutlich ineinander, sodass es sehr schwierig sein dürfte, klare Entscheidungsgrenzen zu finden.

Auch die statistischen Kennwerte sprechen dagegen, nur zwei Dimensionen zu verwenden.
Durch die übrig gebliebenen Merkmale, werden nur $28.5\%$ der ursprünglichen Varianz abgedeckt.
Diese Reduktion der Information ist hier vermutlich zu hoch, um einem Klassifizierungsalgorithmus noch eine gute Datenbasis zu liefern.

#### Beispiel: Dimensionsreduktion von 64 auf 3 Dimensionen mittels PCA

Während mehr als $3$ Dimemnsionen schlecht visualiserbar sind, können wir eine PCA mit $3$ Dimensionen versuchen und darstellen, um eine Intuition zwischen erzielbarer Varianz und Dimensionen zu bekommen. Wir reduzieren also die Dimensionen unserer Origianldaten erneut, diesem Mal auf $3$ Dimensionen:

In [ ]:
pca3D = PCA(3)  # project from 64 to 3 dimensions
projected3D = pca3D.fit_transform(digits.data)
print(digits.data.shape)
print(projected3D.shape)
print("Der Datensatz enthält noch %.1f%c der ursprünglichen Information (Varianz)"
      % (100*sum(pca3D.explained_variance_ratio_), '%'))

Die folgende Visualisierung zeigt, dass die Daten (Punktwolken) bei drei Dimensionen möglicherweise schon besser getrennt werden können als zuvor mit $2$ Dimensionen, wir haben aber nich immer überlappungen.

Durch die übrig gebliebenen Merkmale, werden nun $40.3\%$ der ursprünglichen Varianz abgedeckt (statt nur $28.5\%$ bei 2 Dimensionen).

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')  # Create 3D axes

sc = ax.scatter(projected3D[:, 0], projected3D[:, 1], projected3D[:, 2],
                c=digits.target, edgecolor='none', alpha=0.5,
                cmap=plt.get_cmap('viridis', 10))

ax.set_xlabel('component 1')
ax.set_ylabel('component 2')
ax.set_zlabel('component 3')
fig.colorbar(sc)

#plt.savefig("pca_digit_3d", dpi=300)
#plt.savefig("pca_digit_3d.svg")

plt.show()

### Die Anzahl der Komponenten bestimmen

Um eine passende Anzahl von Hauptkomponenten festzulegen, kann uns wieder die allgmeine PCA helfen.
Hierbei werden ja alle Hauptkomponenten gefunden und nach ihrer Varianz absteigend sortiert.
Wir betrachten nun die kummulative, relative Varianz.
Beginnend mit der *wichtigsten* Hauptkomponente summieren wir den Anteil, den die Komponenten an der gesamten Varianz des Datensatzes abbilden, auf.
So entsteht die unten abgebildete Kurve.
Wir sehen, wie viel uns das hinzunehmen einer Weiteren Komponte zum Modell *bringt*.
Um z.B. 90% der Varianz zu erhalten, benötig man ca. 20 Hauptkomponenten:

In [ ]:
pca = PCA().fit(digits.data)
plt.plot(np.cumsum(pca.explained_variance_ratio_),'o')
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')

#plt.savefig("pca_digit_comp.png", dpi=300)
#plt.savefig("pca_digit_comp.svg")

plt.show()

## PCA zur Rauschunterdrückung

Die Hauptkomponentenzerlegung kann auch verwendet werden, um verrauschte Daten aufzubereiten.
Die Idee ist wie folgt:
Komponenten mit einer Varianz die höher als die des Rauschens ist, sollten durch das Rauschen kaum beeinträchtigt sein.
Wenn wir also nur diese wichtigen Hauptkomponenten werwenden, sollte das Rauschen nach dem wiederherstellen der Daten zu großen Teilen eliminiert sein.

Wir nehmen wieder den Ziffern Datensatz und legen über die Bilder ein künstliches Rauschen:

In [ ]:
def plot_digits(data):
    fig, axes = plt.subplots(4, 10, figsize=(10, 4),
                             subplot_kw={'xticks':[], 'yticks':[]},
                             gridspec_kw=dict(hspace=0.1, wspace=0.1))
    for i, ax in enumerate(axes.flat):
        ax.imshow(data[i].reshape(8, 8),
                  cmap='binary', interpolation='nearest',
                  clim=(0, 16))
    #plt.savefig("pca_digit_noise.png", dpi=300)
    #plt.savefig("pca_digit_noise.svg")
    
    plt.show()

plot_digits(digits.data)

Nun fügen wir den Bilddaten ein Gauß'sches Rauschen hinzu:

In [ ]:
np.random.seed(42)
noisy = np.random.normal(digits.data, 4)
plot_digits(noisy)

Man sieht, dass die Bilder nun deutlich *verpixelt* sind.
Auf diesen Daten, berechnen wir nun ein PCA Modell, dass noch $50\%$ der Varianz erhalten soll:

In [ ]:
pca = PCA(0.50).fit(noisy)
pca.n_components_

Um die $50\%$ der Varianz zu erreichen, müssen $12$ Hauptkomponenten verwendet werden.
Wir berechnen nun diese Hauptkomponenten und stellen mit diesen die Bilddaten über eine inverse Transformation wieder her:

In [ ]:
components = pca.transform(noisy)
filtered = pca.inverse_transform(components)
plot_digits(filtered)

Wie man sieht, ist das Ruaschen deutlich unterdrückt.
Die Bilder wirken zwar etwas *weich gezeichnet*, aber die Konturen der ursprünglichen Ziffern sind deutlicher zu erkennen, als bei den verpixelten Bildern.

# Laden und Vorverarbeiten von Audiodaten für die Klassifikation

Im Folgenden laden wir Audiodaten der Ziffern **0** und **5** aus WAV-Dateien, die im angegebenen Verzeichnis gespeichert sind. Die Audiodateien werden normalisiert und auf eine einheitliche Länge gebracht, damit sie für die anschließende Klassifikation verwendet werden können.


In [ ]:
path = './audio_data/data/**/'  # path to dataset

In [ ]:
def load_examples(path, X):
    
    audiofiles = glob.glob(path, recursive=True)
    for filepath in audiofiles:
        x, fs = sf.read(filepath)
        x = x / np.max(np.abs(x))
        X.append(x)
    
    return X, fs


X = list()

X_0, fs = load_examples(path + '0_*.wav', [])
X_5, _ = load_examples(path + '5_*.wav', [])

X = X_0 + X_5  # alle Beispiele zusammenführen

# Anzahl Beispiele der jeweiligen Klassen
n_0 = len(X_0)
n_5 = len(X_5)

# Zielvariable explizit setzen
y = np.array([0] * n_0 + [1] * n_5)

# Beispiel-Längen ermitteln, um auf gleiche Länge zu bringen (Nullpadding)
lengths = [len(x) for x in X]
N = max(lengths)

X = [np.concatenate((sample, np.zeros(N - len(sample)))) for sample in X]
X = np.array(X)


In [ ]:
print(f'Anzahl 0er Beispiele: {n_0}')
print(f'Anzahl 5er Beispiele: {n_5}')

print('Total number of examples: {}'.format(len(X)))
print('Number of samples per example: {}'.format(N))

Dieser Code zeigt, wie man verrauschte Audiodaten mit PCA wieder sauber macht. Zuerst wird künstliches Rauschen zum Originalsignal hinzugefügt. Danach wird mit PCA das Frequenzspektrum des Signals analysiert und gefiltert, um das Rauschen zu reduzieren.

Zum Vergleich werden für das Original, das verrauschte und das denoisierte Signal jeweils die Wellenform, das Spektrogramm und die MFCCs dargestellt. 

```MFCCs (Mel-Frequency Cepstral Coefficients)``` sind Merkmale, die wichtige Informationen über die Klangfarbe und Struktur eines Audiosignals zusammenfassen und oft in der Sprach- und Audioverarbeitung verwendet werden.


In [ ]:

# PCA-basierte Denoising-Funktion
def denoise_audio(audio, n_components=10):  # Verminderte Anzahl der Komponenten
    stft = librosa.stft(audio)
    magnitude, phase = np.abs(stft), np.angle(stft)

    pca = PCA(n_components=n_components)
    transformed = pca.fit_transform(magnitude.T)
    reconstructed = pca.inverse_transform(transformed).T

    stft_denoised = reconstructed * np.exp(1j * phase)
    denoised_audio = librosa.istft(stft_denoised)
    return denoised_audio

# Funktion zum Hinzufügen von Rauschen
def add_noise(audio, noise_factor=0.005):
    noise = np.random.randn(len(audio))
    noisy_audio = audio + noise_factor * noise
    return noisy_audio

# Funktion zum Zeichnen der Plots
def plot_audio_examples(X, fs):
    for i, audio in enumerate(X):
        plt.figure(figsize=(15, 10))

        # 1. Original Audio
        plt.subplot(3, 3, 1)
        librosa.display.waveshow(audio, sr=fs)
        plt.title(f'Original Waveform')
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        # Spectrogram der Original-Version
        plt.subplot(3, 3, 2)
        D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
        librosa.display.specshow(D, sr=fs, x_axis='time', y_axis='log')
        plt.title('Original Spectrogram')
        plt.colorbar(format='%+2.0f dB')
        plt.set_cmap("viridis")
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        # MFCC der Original-Version
        plt.subplot(3, 3, 3)
        mfccs = librosa.feature.mfcc(y=audio, sr=fs, n_mfcc=13)
        librosa.display.specshow(mfccs, sr=fs, x_axis='time')
        plt.ylabel('Frequency')

        plt.title('Original MFCC')
        plt.colorbar()
        plt.set_cmap("viridis")
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))


        # 2. Noisy Audio
        noisy_audio = add_noise(audio)

        plt.subplot(3, 3, 4)
        librosa.display.waveshow(noisy_audio, sr=fs)
        plt.title(f'Noisy Waveform')
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        plt.subplot(3, 3, 5)
        D_noisy = librosa.amplitude_to_db(np.abs(librosa.stft(noisy_audio)), ref=np.max)
        librosa.display.specshow(D_noisy, sr=fs, x_axis='time', y_axis='log')
        plt.title('Noisy Spectrogram')
        plt.colorbar(format='%+2.0f dB')
        plt.set_cmap("viridis")
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        plt.subplot(3, 3, 6)
        mfccs_noisy = librosa.feature.mfcc(y=noisy_audio, sr=fs, n_mfcc=13)
        librosa.display.specshow(mfccs_noisy, sr=fs, x_axis='time')
        plt.ylabel('Frequency')

        plt.title('Noisy MFCC')
        plt.colorbar()
        plt.set_cmap("viridis")
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))


        # 3. Denoised Audio
        denoised_audio = denoise_audio(noisy_audio)

        plt.subplot(3, 3, 7)
        librosa.display.waveshow(denoised_audio, sr=fs)
        plt.title(f'Denoised Waveform')
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        plt.subplot(3, 3, 8)
        D_denoised = librosa.amplitude_to_db(np.abs(librosa.stft(denoised_audio)), ref=np.max)
        librosa.display.specshow(D_denoised, sr=fs, x_axis='time', y_axis='log')
        plt.title('Denoised Spectrogram')
        plt.colorbar(format='%+2.0f dB')
        plt.set_cmap("viridis")
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        plt.subplot(3, 3, 9)
        mfccs_denoised = librosa.feature.mfcc(y=denoised_audio, sr=fs, n_mfcc=13)
        librosa.display.specshow(mfccs_denoised, sr=fs, x_axis='time')
        plt.ylabel('Frequency')

        plt.title('Denoised MFCC')
        plt.colorbar()
        plt.set_cmap("viridis")
        plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        plt.tight_layout()
        
        # plt.savefig("spectro.png", dpi=300)
        # plt.savefig("spectro.svg")
    
        plt.show()

plot_audio_examples(X_0[0:1], fs)


## Extraktion von MFCC-Features aus den Audiodaten

Um die Audiodaten für die Klassifikation besser nutzbar zu machen, extrahieren wir sogenannte **[Mel-Frequency Cepstral Coefficients (MFCCs)](https://librosa.org/doc/0.11.0/generated/librosa.feature.mfcc.html)**. Diese Features sind kompakte Darstellungen der Klangcharakteristik und werden häufig in der Sprach- und Audiosignalverarbeitung eingesetzt.


In [ ]:
MFCC = [mfcc(y=sample, sr=fs, htk=True) for sample in X]
MFCC = np.array(MFCC)
MFCC = MFCC.reshape((-1, np.prod(MFCC.shape[1:])))

print('Gesamtanzahl der Beispiele: {}'.format(MFCC.shape[0]))
print('Dimension der Merkmalsvektoren: {}'.format(MFCC.shape[1]))

Zunächst zentrieren wir die MFCC-Daten, da PCA voraussetzt, dass die Merkmale einen Mittelwert von Null haben. Anschließend berechnen wir die Hauptkomponenten mit PCA.

Ein **Scree-Plot** ist ein Diagramm, das die Eigenwerte (bzw. Singularwerte) der Hauptkomponenten in absteigender Reihenfolge darstellt. Jede Komponente erklärt einen Teil der Gesamtvarianz in den Daten:

- **X-Achse:** Nummer der Hauptkomponente (Komponentenindex)  
- **Y-Achse:** Eigenwert oder Singularwert (Maß für die erklärte Varianz der Komponente)

Der Scree-Plot hilft dabei, die "wichtigen" Komponenten von den unwichtigen zu unterscheiden. Große Eigenwerte bedeuten, dass die entsprechende Komponente viel Varianz erklärt und somit wichtige Informationen enthält. Kleine Eigenwerte zeigen Komponenten, die vor allem Rauschen oder wenig bedeutende Details erfassen.

Im Scree-Plot sucht man oft nach dem "Knick" (Elbow), also dem Punkt, ab dem die Eigenwerte deutlich flacher verlaufen. Komponenten nach diesem Knick tragen nur noch wenig zur Erklärung der Varianz bei und können häufig weggelassen werden, um die Dimensionen zu reduzieren, ohne viel Information zu verlieren.


In [ ]:
# Daten zentrieren (PCA erwartet Merkmale mit Mittelwert Null)
MFCC_mean = np.mean(MFCC, axis=0)
MFCC_centered = MFCC - MFCC_mean

# PCA auf zentrierten MFCC-Daten
pca = PCA().fit(MFCC_centered)

# Scree-Plot: die ersten 25 Singularwerte anzeigen
def scree_plot(S, N=25):
    
    line, = plt.plot(S[:N], marker='s', markerfacecolor='none')
    plt.xlabel(r'Komponentenindex $r$')
    plt.ylabel(r'Singularwert $\sigma_r$')
    plt.title(f'Scree-Plot der ersten {N} Komponenten')
    
    # plt.savefig("scree_plot.png", dpi=300)
    # plt.savefig("scree_plot.svg")

    plt.grid(True)
    return line

# Plot
scree_plot(pca.singular_values_, N=25)
plt.show()


Wir führen PCA auf den zentrierten MFCC-Daten durch und analysieren anschließend die kumulative erklärte Varianz.

Der Plot zeigt die **kumulative erklärte Varianz** in Abhängigkeit von der Anzahl der Hauptkomponenten. Die kumulative erklärte Varianz gibt an, wie viel Prozent der Gesamtvarianz der Daten durch die ersten *n* Komponenten zusammen erklärt wird.


In [ ]:
pca = PCA().fit(MFCC_centered)

plt.plot(np.cumsum(pca.explained_variance_ratio_)[:50])
plt.xlabel('Anzahl Hauptkomponenten')
plt.ylabel('Kumulative erklärte Varianz')
plt.title('Erklärte Varianz der PCA auf MFCCs (erste 50 Komponenten)')
plt.grid(True)

# plt.savefig("pca_mfcc_varianz.png", dpi=300)
# plt.savefig("pca_mfcc_varianz.svg")

plt.show()


Wir führen eine PCA-Reduktion der MFCC-Daten auf 6 Dimensionen durch. Zunächst wird das PCA-Modell mit `n_components=6` erstellt und auf die zentrierten Daten angepasst. Anschließend werden die Daten auf den neuen, reduzierten 6-dimensionalen Merkmalsraum abgebildet.

Das Ergebnis ist ein Datensatz mit derselben Anzahl an Beispielen, aber mit deutlich reduzierter Dimensionalität.


In [ ]:
# PCA-Reduktion auf 6 Dimensionen
pca = PCA(n_components=6)
Xr = pca.fit_transform(MFCC_centered)

print('Gesamtanzahl der Beispiele: {}'.format(Xr.shape[0]))
print('Dimension der reduzierten Merkmalsvektoren: {}'.format(Xr.shape[1]))

Wir haben die Daten mittels PCA auf 6 Dimensionen reduziert. Im Folgenden visualisieren wir die Verteilung der beiden Klassen (Ziffer 0 und Ziffer 5) in den ersten sechs Hauptkomponenten.

Dazu werden drei Scatterplots erstellt, die jeweils die Paare von Hauptkomponenten (PC1 vs. PC2, PC3 vs. PC4, PC5 vs. PC6) darstellen. Die Punkte der Klasse 0 sind blau, die der Klasse 5 rot markiert.

Diese Darstellung zeigt, wie gut sich die beiden Ziffern im reduzierten Merkmalsraum separieren lassen und gibt Einblick in die Trennbarkeit der Klassen nach der Dimensionsreduktion.


In [ ]:
print('Anzahl der Beispiele:', Xr.shape[0])
print('Dimension der reduzierten Merkmalsvektoren:', Xr.shape[1])

# Zielwerte: 0 = Ziffer 0, 1 = Ziffer 5
y = np.array([0] * n_0 + [1] * n_5)

# Plot: 3 Scatterplots (PC1 vs PC2, PC3 vs PC4, PC5 vs PC6)
fig = plt.figure(figsize=(10, 3), constrained_layout=True)
gs = fig.add_gridspec(1, 3, wspace=0.3)

for n in range(3):
    ax = fig.add_subplot(gs[n])
    ax.scatter(Xr[y==0, 2*n], Xr[y==0, 2*n+1], color='blue', label='Ziffer 0', s=5)
    ax.scatter(Xr[y==1, 2*n], Xr[y==1, 2*n+1], color='red', label='Ziffer 5', s=5)
    ax.set_xlabel(f'PC{2*n + 1}')
    ax.set_ylabel(f'PC{2*n + 2}')
    ax.set_title(f'PC{2*n + 1} vs. PC{2*n + 2}')
    ax.grid(True)
    ax.legend()

#plt.savefig("scatter_audio.png", dpi=300)
#plt.savefig("scatter_audio.svg")

plt.suptitle('Scatterplots der ersten 6 Hauptkomponenten (PCA)', fontsize=14)
plt.show()


Wir verwenden die ersten beiden Hauptkomponenten (PC1 und PC2) als Merkmale für eine logistische Regression, um die Klassifikation der Ziffern 0 und 5 durchzuführen.

Das Modell wird mit diesen reduzierten Daten trainiert, um zu prüfen, wie gut sich die beiden Klassen mit nur zwei Hauptkomponenten trennen lassen.


In [ ]:
X_plot = Xr[:, :2]  # nur PC1 und PC2

clf = LogisticRegression()
clf.fit(X_plot, y)

Wir evaluieren die Leistung des trainierten logistischen Regressionsmodells anhand der Genauigkeit (Accuracy) auf den Trainingsdaten.

Der Plot zeigt die Entscheidunggrenze des Modells im Raum der ersten beiden Hauptkomponenten (PC1 und PC2). Die Farbflächen geben die von der Klassifikation vorhergesagten Klassenbereiche an, während die Punkte die tatsächlichen Datenpunkte der Ziffern 0 (blau) und 5 (rot) darstellen.

Die angegebene Accuracy gibt an, wie gut das Modell die Trainingsdaten korrekt klassifiziert hat.


In [ ]:
from sklearn.metrics import accuracy_score

# Vorhersagen auf Trainingsdaten
y_pred = clf.predict(X_plot)
acc = accuracy_score(y, y_pred)

plt.figure(figsize=(8,6))

# Hintergrund mit Farben für Klassen
plt.contourf(xx, yy, Z, alpha=0.2, colors=['blue', 'red'])

# Scatterplot der Datenpunkte
plt.scatter(X_plot[y==0, 0], X_plot[y==0, 1], color='blue', label='Ziffer 0', s=20)
plt.scatter(X_plot[y==1, 0], X_plot[y==1, 1], color='red', label='Ziffer 5', s=20)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'Lineare Klassifikation auf PC1 und PC2\nAccuracy: {acc:.2f}')
plt.legend()
plt.grid(True)

#plt.savefig("lineare_klassifikation.png", dpi=300)
#plt.savefig("lineare_klassifikation.svg")

plt.show()


# Rauschunterdrückung mittels PCA

Im nächsten Schritt wollen wir untersuchen, wie sich Rauschen auf unsere MFCC-Daten auswirkt und wie wir dieses mithilfe von Principal Component Analysis (PCA) reduzieren können.

Zunächst betrachten wir einige Original-MFCC-Beispiele der Ziffern 0 und 5. Diese geben uns einen Eindruck von den charakteristischen Mustern in den jeweiligen Klassen.


In [ ]:
def plot_samples_per_digit_lineplot_ordered(X, digit_start_index, n_samples=4):
    samples = X[digit_start_index:digit_start_index + n_samples]
    
    fig, axes = plt.subplots(1, n_samples, figsize=(n_samples * 3, 3),
                             subplot_kw={'xticks':[], 'yticks':[]})
    
    for i, ax in enumerate(axes):
        ax.plot(samples[i])
        ax.set_title(f'Ziffer {0 if digit_start_index == 0 else 5}')
        ax.grid(True)
        
    plt.tight_layout()
    
    #plt.savefig("audio_pca.png", dpi=300)
    #plt.savefig("audio_pca.svg")

    plt.show()

# Beispielaufrufe
plot_samples_per_digit_lineplot_ordered(MFCC, digit_start_index=0, n_samples=4)       # Ziffer 0
plot_samples_per_digit_lineplot_ordered(MFCC, digit_start_index=n_0, n_samples=4)     # Ziffer 5


Nun fügen wir den Bilddaten ein Gauß'sches Rauschen hinzu:

In [ ]:
np.random.seed(42)
MFCC_noisy = MFCC + 6 * np.random.normal(size=MFCC.shape)
plot_samples_per_digit_lineplot_ordered(MFCC_noisy, digit_start_index=0, n_samples=4)
plot_samples_per_digit_lineplot_ordered(MFCC_noisy, digit_start_index=n_0, n_samples=4)



Man sieht, dass die Audio-Signale durch das Hinzufügen von Rauschen verzerrt sind.
Auf diesen verrauschten MFCC-Daten berechnen wir nun ein PCA-Modell, das mindestens 50% der Varianz erhält,
um die wesentlichen Merkmale zu extrahieren und das Rauschen zu reduzieren.


In [ ]:
pca = PCA(0.50).fit(MFCC_noisy)
pca.n_components_

Um mindestens $50\%$ der Varianz zu erhalten, müssen 3 Hauptkomponenten verwendet werden.
Wir berechnen diese Hauptkomponenten auf den MFCC-Daten und rekonstruieren die Signale
durch inverse Transformation, um Rauschen zu reduzieren und wichtige Merkmale zu erhalten.


In [ ]:
MFCC_transformed = pca.transform(MFCC_noisy)
MFCC_reconstructed = pca.inverse_transform(MFCC_transformed)
plot_samples_per_digit_lineplot_ordered(MFCC, digit_start_index=0, n_samples=4) # Original
plot_samples_per_digit_lineplot_ordered(MFCC_reconstructed, digit_start_index=0, n_samples=4) # Rekonstruiert


Zunächst fügen wir den MFCC-Daten Rauschen hinzu und wenden dann PCA an, um die Daten mit 50% der Varianz zu rekonstruieren. 

Der folgende Plot zeigt im direkten Vergleich die Original-MFCCs (oben) und die rekonstruierten MFCCs nach der PCA-Denoising-Methode (unten).


**Quelle:**

[1] Jake VanderPlas, [*Python Data Science Handbook*](https://jakevdp.github.io/PythonDataScienceHandbook), O'Reilly, 2016